# Proyecto Final — TC2032 Agentes Inteligentes
## Coordinación inteligente de cruces semafóricos: corredor **Calle Libertad** (Col. Nuevo Repueblo, Monterrey)

**Caso de estudio:** corredor de la calle **Libertad** (un sentido, Oeste→Este) con sus tres
intersecciones semaforizadas: **Libertad × Sonora**, **Libertad × Chiapas** y **Libertad × Tepic**.

**Qué se compara (los dos escenarios del proyecto):**
- **Escenario 1 — Sin coordinación:** los tres semáforos operan con el mismo ciclo pero de forma
  independiente (offset 0 en todos).
- **Escenario 2 — Con coordinación (onda verde):** mismos ciclos, pero cada semáforo desfasa su
  verde un *offset* igual al tiempo de viaje desde el cruce anterior, de modo que un pelotón que
  arranca en Sonora encuentra verde en Chiapas y Tepic.

**Método de decisión (heurística):** los agentes semáforo siguen una regla de ciclo fijo con
reparto de verde proporcional a la demanda, y la coordinación usa la heurística clásica de onda
verde: `offset_i = distancia_i / velocidad`.

Ejecutar con `Entorno de ejecución → Ejecutar todo`. Reproducible con `seed = 42`.

In [ ]:
# === 1. Librerías y configuración ===
import random
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation, rc
from matplotlib.patches import Patch, Rectangle

rc("animation", html="jshtml")     # reproductor de animacion embebido en Colab
SEED = 42
print("Listo.")

## 2. Entorno — red vial del corredor
La red se modela como una cuadrícula discreta: **Libertad** es una fila horizontal de un solo
sentido (O→E, como en el mapa) y las tres transversales son columnas de un sentido
(supuesto: Sonora baja, Chiapas sube, Tepic baja, patrón típico de calles alternadas del centro).
Las celdas donde una transversal toca Libertad son las **zonas de intersección**, controladas por
un semáforo. La separación entre cruces es uniforme (14 celdas), aproximando las cuadras
similares que se ven en el mapa real.

In [ ]:
# === 2. Entorno: corredor + 3 transversales ===
W, H = 46, 13                       # ancho y alto de la cuadricula
ROW_MAIN = 6                        # fila de la calle Libertad (un sentido O->E)
X_INT = [8, 22, 36]                 # columnas de las intersecciones (separacion = 14 celdas)
CROSS_DIR = {8: 1, 22: -1, 36: 1}   # sentido de cada transversal: +1 baja, -1 sube
NAMES = {8: "Sonora", 22: "Chiapas", 36: "Tepic"}

def is_street(r, c):
    """Celda transitable: la fila de Libertad o la columna de una transversal."""
    return r == ROW_MAIN or c in X_INT

# Fondo para dibujar (claro = calle, oscuro = manzana)
bg = np.full((H, W), 0.18)
for r in range(H):
    for c in range(W):
        if is_street(r, c):
            bg[r, c] = 0.78

print(f"Red de {W}x{H}. Libertad en fila {ROW_MAIN}. Cruces en columnas {X_INT}:",
      [NAMES[x] for x in X_INT])

## 3. Agentes
**Agente semáforo** (`TrafficLight`): observa el reloj global y su *offset*; su estado es la fase
(`verde Libertad` o `verde transversal`) y su decisión es qué fase mostrar en cada paso, según un
ciclo fijo `C = 24` con verde principal `G = 14` (reparto ≈ demanda 0.40 vs 0.15).
**Agente vehículo** (`Vehicle`): estado = posición, dirección, ruta; percibe la celda de enfrente
y el semáforo de su aproximación; decide avanzar o detenerse. Registra sus métricas (tiempo de
viaje, pasos detenido, número de paradas).

In [ ]:
# === 3. Agentes: semaforo y vehiculo ===
class TrafficLight:
    """Agente semaforo de 2 fases con ciclo fijo y offset (heuristica de coordinacion)."""
    def __init__(self, x, cycle, green_main, offset):
        self.x = x                  # columna del cruce que controla
        self.cycle = cycle          # duracion del ciclo (pasos)
        self.green_main = green_main# pasos de verde para Libertad dentro del ciclo
        self.offset = offset        # desfase del ciclo (la clave de la onda verde)
    def main_green(self, t):
        """Decision del agente: True si Libertad tiene verde en el paso t."""
        return (t + self.offset) % self.cycle < self.green_main

class Vehicle:
    """Agente vehiculo: avanza 1 celda/paso si su celda siguiente esta libre y con verde."""
    _id = 0
    def __init__(self, r, c, d, route):
        self.id = Vehicle._id; Vehicle._id += 1
        self.r, self.c, self.d = r, c, d   # posicion (fila, col) y direccion (dr, dc)
        self.route = route                 # "main" (Libertad) o nombre de la transversal
        self.state = "moving"              # "moving" / "done"
        self.spawn_t = None; self.exit_t = None
        self.wait_steps = 0                # pasos que paso detenido
        self.stops = 0                     # numero de detenciones (eventos parar-arrancar)
        self.was_stopped = False
    @property
    def pos(self): return (self.r, self.c)
    def nxt(self): return (self.r + self.d[0], self.c + self.d[1])

print("Agentes definidos.")

## 4. Entorno dinámico y reglas de interacción
Reglas del vehículo (locales y simples): **(a)** si la celda siguiente está ocupada → detenerse;
**(b)** si la celda siguiente es una zona de intersección y su semáforo está en rojo para su
aproximación → detenerse; **(c)** en cualquier otro caso → avanzar una celda. La ocupación se
actualiza al instante, de modo que dos vehículos jamás comparten celda (cero colisiones).

In [ ]:
# === 4. Entorno dinamico (CorridorEnv) ===
class CorridorEnv:
    def __init__(self, offsets, cycle=24, green_main=14, p_main=0.40, p_cross=0.15, seed=SEED):
        random.seed(seed); Vehicle._id = 0
        self.t = 0
        self.lights = [TrafficLight(x, cycle, green_main, o) for x, o in zip(X_INT, offsets)]
        self.light_by_x = {l.x: l for l in self.lights}
        self.p_main, self.p_cross = p_main, p_cross   # demanda (prob. de llegada por paso)
        self.veh = []                                  # agentes activos
        self.spawned = 0; self.exited = 0
        self.done_vehicles = []                        # agentes que ya salieron (para metricas)
        self.queue_hist = {x: [] for x in X_INT}       # cola en Libertad antes de cada cruce

    def occ(self): return {v.pos for v in self.veh if v.state == "moving"}
    def ing(self, r, c): return 0 <= r < H and 0 <= c < W

    def spawn(self, occ):
        """Llegadas aleatorias: Libertad por el Oeste; cada transversal por su extremo."""
        entries = [(ROW_MAIN, 0, (0, 1), "main", self.p_main)]
        for x in X_INT:
            r0 = 0 if CROSS_DIR[x] == 1 else H - 1
            entries.append((r0, x, (CROSS_DIR[x], 0), NAMES[x], self.p_cross))
        for r, c, d, route, p in entries:
            if random.random() < p and (r, c) not in occ:
                v = Vehicle(r, c, d, route); v.spawn_t = self.t
                self.veh.append(v); occ.add((r, c)); self.spawned += 1

    def light_blocks(self, v, nr, nc):
        """True si el semaforo en rojo impide a v entrar a la celda de cruce (nr, nc)."""
        if nr == ROW_MAIN and nc in X_INT:
            if v.route == "main":
                return not self.light_by_x[nc].main_green(self.t)   # rojo para Libertad
            return self.light_by_x[nc].main_green(self.t)           # rojo para transversal
        return False

    def step(self):
        """Un paso: 1) llegadas, 2) movimiento con las reglas (a)-(c), 3) metricas."""
        occ = self.occ(); self.spawn(occ)
        moved, progress = set(), True
        while progress:                       # pasadas hasta punto fijo (avance por caravana)
            progress = False
            for v in self.veh:
                if v.state != "moving" or v.id in moved: continue
                nr, nc = v.nxt()
                if not self.ing(nr, nc):                    # sale de la red
                    v.state = "done"; v.exit_t = self.t; self.exited += 1
                    self.done_vehicles.append(v)
                    occ.discard(v.pos); moved.add(v.id); progress = True; continue
                if (is_street(nr, nc) and (nr, nc) not in occ
                        and not self.light_blocks(v, nr, nc)):
                    occ.discard(v.pos); v.r, v.c = nr, nc; occ.add(v.pos)  # ocupa al instante
                    moved.add(v.id); progress = True
        for v in self.veh:                    # espera y numero de paradas por vehiculo
            if v.state == "moving" and v.id not in moved:
                v.wait_steps += 1
                if not v.was_stopped: v.stops += 1; v.was_stopped = True
            elif v.id in moved:
                v.was_stopped = False
        for x in X_INT:                       # longitud de cola aguas arriba de cada cruce
            q = sum(1 for v in self.veh if v.state == "moving" and v.route == "main"
                    and v.r == ROW_MAIN and x - 8 <= v.c < x and v.id not in moved)
            self.queue_hist[x].append(q)
        self.veh = [v for v in self.veh if v.state == "moving"]
        self.t += 1

    def snap(self):
        return ([(v.r, v.c, v.route) for v in self.veh],
                [l.main_green(self.t) for l in self.lights])

def run(offsets, steps=300, seed=SEED, **kw):
    env = CorridorEnv(offsets, seed=seed, **kw)
    hist = [env.snap()]
    for _ in range(steps):
        env.step(); hist.append(env.snap())
    return env, hist

def metrics(env):
    """Indicadores agregados por escenario."""
    main_done  = [v for v in env.done_vehicles if v.route == "main"]
    cross_done = [v for v in env.done_vehicles if v.route != "main"]
    avg = lambda xs: sum(xs) / len(xs) if xs else 0
    return {
        "Vehiculos generados": env.spawned,
        "Vehiculos atendidos (salieron)": env.exited,
        "Tiempo medio de viaje en Libertad (pasos)": round(avg([v.exit_t - v.spawn_t for v in main_done]), 1),
        "Tiempo medio de espera en Libertad (pasos)": round(avg([v.wait_steps for v in main_done]), 1),
        "Paradas promedio por vehiculo (Libertad)": round(avg([v.stops for v in main_done]), 2),
        "Tiempo medio de espera transversales (pasos)": round(avg([v.wait_steps for v in cross_done]), 1),
        "Cola maxima Sonora": max(env.queue_hist[8]),
        "Cola maxima Chiapas": max(env.queue_hist[22]),
        "Cola maxima Tepic": max(env.queue_hist[36]),
        "Cola promedio Chiapas": round(avg(env.queue_hist[22]), 2),
        "Cola promedio Tepic": round(avg(env.queue_hist[36]), 2),
    }

print("Entorno dinamico listo.")

## 5. Demanda vehicular
> **NOTA PARA EL EQUIPO:** aquí se conectan los **datos de demanda del curso**. Sustituyan
> `p_main` y `p_cross` por las tasas derivadas de su tabla de demanda (vehículos/hora →
> probabilidad de llegada por paso de simulación) y documenten la conversión en el reporte.

In [ ]:
# === 5. Parametros de demanda (EDITAR con los datos del curso) ===
P_MAIN  = 0.40   # prob. de llegada por paso en Libertad  (demanda alta del corredor)
P_CROSS = 0.15   # prob. de llegada por paso en cada transversal (demanda baja)
CYCLE, GREEN_MAIN = 24, 14   # ciclo comun y reparto de verde (~prop. a la demanda)
STEPS = 300
print(f"Demanda: Libertad={P_MAIN}, transversales={P_CROSS} | Ciclo={CYCLE}, verde Libertad={GREEN_MAIN}")

## 6. Escenario 1 — Operación **sin coordinación**
Los tres semáforos usan el mismo ciclo con **offset 0** (operación independiente): el verde inicia
al mismo tiempo en los tres cruces, pero como el pelotón tarda 14 pasos en llegar al siguiente
cruce, llega ya en rojo y se detiene una y otra vez.

In [ ]:
# === 6. Escenario 1: sin coordinacion ===
env1, hist1 = run(offsets=[0, 0, 0], steps=STEPS, p_main=P_MAIN, p_cross=P_CROSS,
                  cycle=CYCLE, green_main=GREEN_MAIN)
m1 = metrics(env1)
for k, v in m1.items(): print(f"  {k}: {v}")

## 7. Escenario 2 — Coordinación por **onda verde**
Heurística de coordinación: `offset_i = distancia al primer cruce / velocidad (mod ciclo)`.
Con separación de 14 celdas y velocidad de 1 celda/paso: offsets `[0, 14, 4]` (28 mod 24 = 4).
El verde "viaja" con el pelotón.

In [ ]:
# === 7. Escenario 2: onda verde ===
OFFSETS_WAVE = [(x - X_INT[0]) % CYCLE for x in X_INT]   # [0, 14, 4]
print("Offsets de onda verde:", OFFSETS_WAVE)
env2, hist2 = run(offsets=OFFSETS_WAVE, steps=STEPS, p_main=P_MAIN, p_cross=P_CROSS,
                  cycle=CYCLE, green_main=GREEN_MAIN)
m2 = metrics(env2)
for k, v in m2.items(): print(f"  {k}: {v}")

In [ ]:
# === 8. Visualizacion del entorno y de un instante de la simulacion ===
ROUTE_COLORS = {"main": "#2a9d8f", "Sonora": "#f4a261", "Chiapas": "#1d4ed8", "Tepic": "#9b5de5"}

def draw_state(ax, snap, title=""):
    vehicles, greens = snap
    ax.clear()
    ax.imshow(bg, cmap="Greys_r", vmin=0, vmax=1, origin="upper", aspect="equal")
    for i, x in enumerate(X_INT):       # zona de interseccion + estado del semaforo
        ax.add_patch(Rectangle((x-0.5, ROW_MAIN-0.5), 1, 1, fill=False,
                               edgecolor="#e9c46a", lw=1.6, zorder=2))
        ax.scatter(x, ROW_MAIN-1.6, marker="s", s=90, zorder=4,
                   c="#21c55d" if greens[i] else "#ef4444", edgecolors="black", linewidths=0.5)
        ax.text(x, H-0.2, NAMES[x], ha="center", fontsize=8, color="white")
    for r, c, route in vehicles:
        ax.scatter(c, r, c=ROUTE_COLORS[route], s=60, edgecolors="black", lw=0.4, zorder=3)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlim(-0.5, W-0.5); ax.set_ylim(H-0.5, -0.5)

LEGEND = ([Patch(facecolor=ROUTE_COLORS[k], edgecolor="black",
                 label=("Libertad (O->E)" if k=="main" else k)) for k in ROUTE_COLORS] +
          [Patch(facecolor="#21c55d", edgecolor="black", label="Verde Libertad"),
           Patch(facecolor="#ef4444", edgecolor="black", label="Rojo Libertad")])

fig, axes = plt.subplots(2, 1, figsize=(11, 6))
draw_state(axes[0], hist1[60], "Escenario 1 (sin coordinacion) - paso 60")
draw_state(axes[1], hist2[60], "Escenario 2 (onda verde) - paso 60")
axes[0].legend(handles=LEGEND, loc="upper left", bbox_to_anchor=(1.01, 1.0), fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# === 9. Animacion (Escenario 2, primeros 120 pasos) ===
figA, axA = plt.subplots(figsize=(10, 3.2))
def update(f):
    draw_state(axA, hist2[f], f"Onda verde - paso {f}")
    return axA
anim = animation.FuncAnimation(figA, update, frames=121, interval=200, blit=False)
plt.close(figA)
anim

In [ ]:
# === 10. Resultados comparativos: tabla, colas y barras ===
print(f"{'Indicador':47s}{'Esc.1 sin coord.':>18s}{'Esc.2 onda verde':>18s}")
for k in m1:
    print(f"{k:47s}{str(m1[k]):>18s}{str(m2[k]):>18s}")

# Series de tiempo de colas en Chiapas y Tepic (donde la onda verde actua)
fig2, axs = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True)
for ax, x in zip(axs, [22, 36]):
    ax.plot(env1.queue_hist[x], color="#ef4444", lw=1.2, label="Sin coordinacion")
    ax.plot(env2.queue_hist[x], color="#21c55d", lw=1.2, label="Onda verde")
    ax.set_title(f"Cola en Libertad antes de {NAMES[x]}"); ax.set_xlabel("paso")
axs[0].set_ylabel("vehiculos en cola"); axs[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

# Barras comparativas de los indicadores clave
keys = ["Tiempo medio de viaje en Libertad (pasos)", "Tiempo medio de espera en Libertad (pasos)",
        "Paradas promedio por vehiculo (Libertad)", "Vehiculos atendidos (salieron)"]
short = ["T. viaje", "T. espera", "Paradas/veh", "Atendidos"]
fig3, ax3 = plt.subplots(figsize=(8, 3.6))
xpos = np.arange(len(keys)); wbar = 0.36
b1 = ax3.bar(xpos - wbar/2, [m1[k] for k in keys], wbar, color="#ef4444",
             edgecolor="black", label="Esc.1 sin coordinacion")
b2 = ax3.bar(xpos + wbar/2, [m2[k] for k in keys], wbar, color="#21c55d",
             edgecolor="black", label="Esc.2 onda verde")
for bars in (b1, b2):
    for b in bars:
        ax3.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f"{b.get_height():g}",
                 ha="center", fontsize=8, fontweight="bold")
ax3.set_xticks(xpos); ax3.set_xticklabels(short); ax3.legend(fontsize=8)
ax3.set_title("Comparacion de indicadores (300 pasos, misma demanda y semilla)")
plt.tight_layout(); plt.show()

## 11. Diagrama espacio-tiempo (bandas de coordinación)
El diagrama espacio-tiempo es la forma clásica de visualizar una onda verde. El eje horizontal es
el tiempo y el vertical la posición a lo largo del corredor; las franjas verdes/rojas son la fase
de cada semáforo y las líneas diagonales son las trayectorias reales de los vehículos de Libertad.
Sin coordinación las trayectorias chocan con las franjas rojas y se aplanan (los autos frenan);
con onda verde las diagonales atraviesan los tres cruces dentro de las franjas verdes.

In [ ]:
# === 11. Diagrama espacio-tiempo: bandas de semáforo + trayectorias reales ===
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

ST_STEPS = 120   # ventana de tiempo a graficar (más corta que STEPS para ver el detalle)

def _trajectories(offsets, seed=SEED):
    """Reejecuta la simulación registrando la columna de cada vehículo del corredor por paso."""
    Vehicle._id = 0
    env = CorridorEnv(offsets, cycle=CYCLE, green_main=GREEN_MAIN,
                      p_main=P_MAIN, p_cross=P_CROSS, seed=seed)
    traj = {}
    for t in range(ST_STEPS):
        env.step()
        for v in env.veh:
            if v.route == "main":
                traj.setdefault(v.id, []).append((t, v.c))
    return traj

def _green(offset, t):
    return (t + offset) % CYCLE < GREEN_MAIN

def draw_space_time(ax, offsets, title):
    ax.set_facecolor("#11131a")
    # Franjas de fase de cada semáforo (verde/rojo) a lo largo del tiempo, en su posición Y
    for x, off in zip(X_INT, offsets):
        for t in range(ST_STEPS):
            ax.plot([t, t + 1], [x, x], lw=7, solid_capstyle="butt", alpha=0.85, zorder=1,
                    color="#21c55d" if _green(off, t) else "#ef4444")
    # Trayectorias reales de los vehículos del corredor (diagonales)
    for pts in _trajectories(offsets).values():
        if len(pts) >= 2:
            ax.plot([p[0] for p in pts], [p[1] for p in pts],
                    color="#67e8f9", lw=1.0, alpha=0.55, zorder=2)
    ax.set_title(title, color="#e8e8e8", fontsize=12, pad=10)
    ax.set_xlabel("Tiempo (pasos)", color="#cbd5e1")
    ax.set_ylabel("Corredor (Oeste → Este)", color="#cbd5e1")
    ax.set_yticks(X_INT); ax.set_yticklabels([NAMES[x] for x in X_INT], color="#cbd5e1")
    ax.tick_params(colors="#94a3b8")
    ax.set_xlim(0, ST_STEPS); ax.set_ylim(0, W)
    for s in ax.spines.values(): s.set_color("#334155")

fig_st, axes_st = plt.subplots(2, 1, figsize=(11, 8))
fig_st.patch.set_facecolor("#0b0d12")
draw_space_time(axes_st[0], [0, 0, 0], "Escenario 1 — Sin coordinación (offsets 0, 0, 0)")
draw_space_time(axes_st[1], OFFSETS_WAVE, f"Escenario 2 — Onda verde (offsets {OFFSETS_WAVE})")
axes_st[0].legend(handles=[
    Patch(facecolor="#21c55d", label="Verde Libertad"),
    Patch(facecolor="#ef4444", label="Rojo Libertad"),
    Line2D([0], [0], color="#67e8f9", lw=2, label="Trayectoria de vehículo")],
    loc="upper right", fontsize=8, facecolor="#1e293b", labelcolor="#e8e8e8", edgecolor="#334155")
plt.tight_layout(); plt.show()

## 12. Gráfica comparativa de indicadores
Barras agrupadas con los indicadores clave de ambos escenarios. Nota: la comparación es
**sin coordinación vs. onda verde** (la heurística implementada en este proyecto); no se usa
Q-Learning. Los valores provienen de la simulación (300 pasos, semilla 42).

In [ ]:
# === 12. Gráfica de barras comparativa (estilo oscuro, datos reales) ===
labels   = ["Vehículos\natendidos", "Tiempo de\nviaje", "Tiempo de\nespera", "Paradas por\nvehículo"]
sin_co   = [m1["Vehiculos atendidos (salieron)"], m1["Tiempo medio de viaje en Libertad (pasos)"],
            m1["Tiempo medio de espera en Libertad (pasos)"], m1["Paradas promedio por vehiculo (Libertad)"]]
onda     = [m2["Vehiculos atendidos (salieron)"], m2["Tiempo medio de viaje en Libertad (pasos)"],
            m2["Tiempo medio de espera en Libertad (pasos)"], m2["Paradas promedio por vehiculo (Libertad)"]]

x = np.arange(len(labels)); wbar = 0.36
fig_b, ax_b = plt.subplots(figsize=(9, 4.6))
fig_b.patch.set_facecolor("#0b0d12"); ax_b.set_facecolor("#11131a")
b1 = ax_b.bar(x - wbar/2, sin_co, wbar, label="Sin coordinación", color="#3b82f6", edgecolor="#0b0d12")
b2 = ax_b.bar(x + wbar/2, onda,  wbar, label="Onda verde",       color="#21c55d", edgecolor="#0b0d12")
for bars in (b1, b2):
    for b in bars:
        ax_b.text(b.get_x() + b.get_width()/2, b.get_height(),
                  f"{b.get_height():g}", ha="center", va="bottom", color="#e8e8e8", fontsize=9, fontweight="bold")
ax_b.set_title("Comparación de coordinación semafórica — corredor Libertad",
               color="#e8e8e8", fontsize=12, pad=10)
ax_b.set_ylabel("Valor", color="#cbd5e1")
ax_b.set_xticks(x); ax_b.set_xticklabels(labels, color="#cbd5e1")
ax_b.tick_params(colors="#94a3b8")
ax_b.legend(facecolor="#1e293b", labelcolor="#e8e8e8", edgecolor="#334155")
ax_b.grid(axis="y", color="#334155", linewidth=0.4, alpha=0.6)
for s in ax_b.spines.values(): s.set_color("#334155")
ax_b.set_axisbelow(True)
plt.tight_layout(); plt.show()

print("Nota: 'Tiempo' y 'Paradas' menores son mejores; 'Vehículos atendidos' mayor es mejor.")

## 13. Conclusión breve del notebook
Con la misma demanda y la misma semilla, la coordinación por onda verde redujo el tiempo medio de
viaje del corredor ≈22%, el tiempo de espera ≈68% y las paradas por vehículo de ~2.6 a ~1.0,
atendiendo además más vehículos, sin aumentar la espera de las calles transversales. El análisis
completo, la justificación de los resultados y las limitaciones se desarrollan en el reporte
técnico (PDF).